# Solution B1 · A complete logistic on the loans: pipeline and ablation study

**Type**: Implement with scikit-learn

## Objective

Build a classification pipeline on mixed data with missing values (numerics with missingness indicators, nominals of low and high cardinality, an ordinal),
measure in cross-validation the **contribution of each piece** with an ablation study and read the coefficients as odds ratios.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler

df = ds.loans()
X, y = df.drop(columns="defaulted"), df["defaulted"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
NUM = ["age", "income", "job_tenure", "amount", "ratio"]


def add_ratio(X):
    X = X.copy()
    X["ratio"] = X["amount"] / X["income"]              # NaN where income is missing: the imputer will handle it
    return X

## Commented solution

In [ ]:
def build(indicators, categoricals):
    groups = [("num", make_pipeline(SimpleImputer(strategy="median", add_indicator=indicators), StandardScaler()), NUM)]
    if categoricals:
        groups.append(("nom", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")),
                       ["city", "contract_type", "occupation"]))
        groups.append(("ord", make_pipeline(SimpleImputer(strategy="most_frequent"), OrdinalEncoder(categories=[ds.EDUCATION_ORDER]), StandardScaler()),
                       ["education"]))
    return make_pipeline(FunctionTransformer(add_ratio), ColumnTransformer(groups, sparse_threshold=0), LogisticRegression(C=1, max_iter=2000))

In [ ]:
variants = {"numerics only": (False, False), "+ indicators": (True, False), "+ categoricals": (False, True), "all": (True, True)}
cv_auc = pd.Series({name: cross_val_score(build(*args), X_tr, y_tr, cv=cv, scoring="roc_auc").mean() for name, args in variants.items()})
indicators_gain = cv_auc["+ indicators"] - cv_auc["numerics only"]
categoricals_gain = cv_auc["+ categoricals"] - cv_auc["numerics only"]
print(cv_auc.round(4).to_string())
print(f"gain of the missingness indicators: {indicators_gain:+.4f} | gain of the categoricals: {categoricals_gain:+.4f}")

In [ ]:
final = build(True, True).fit(X_tr, y_tr)
test_auc = roc_auc_score(y_te, final.predict_proba(X_te)[:, 1])
names = final[1].get_feature_names_out()
coef = pd.Series(final[-1].coef_[0], index=names)
scale = final[1].named_transformers_["num"][-1].scale_                                     # standard deviation used for every column of the numeric block
k = list(names).index("num__missingindicator_income")
missing_income_odds = float(np.exp(coef.iloc[k] / scale[k]))                               # from "per 1 std dev" to "missing versus present"
num = coef[[n for n in names if n.startswith("num__")]]
top4_numeric = list(num.abs().nlargest(4).index)
print(f"test AUC: {test_auc:.4f} | odds ratio of income missing (versus present): {missing_income_odds:.2f}")
print(num.round(3).to_string())

## Check

In [ ]:
with verify():
    check(list(cv_auc.index) == ["numerics only", "+ indicators", "+ categoricals", "all"], "four variants in the right order", "index: numerics only, + indicators, + categoricals, all")
    check(cv_auc["all"] > 0.80 and cv_auc["all"] == cv_auc.max(), "the complete variant is the best (AUC > 0.80)", "all = indicators + categoricals")
    check(close(indicators_gain, cv_auc["+ indicators"] - cv_auc["numerics only"]) and indicators_gain > 0.04,
          "the missingness indicators bring over 0.04 of AUC", "add_indicator=True in the SimpleImputer of the numerics")
    check(close(categoricals_gain, cv_auc["+ categoricals"] - cv_auc["numerics only"]) and 0.01 < categoricals_gain < indicators_gain,
          "the categoricals contribute less than the indicators", "categoricals_gain must be positive but lower than indicators_gain")
    check(abs(test_auc - cv_auc["all"]) < 0.06 and close(test_auc, roc_auc_score(y_te, final.predict_proba(X_te)[:, 1])), "test AUC consistent with the cross-validated one", "roc_auc_score on the test set with predict_proba")
    check(missing_income_odds is not None and missing_income_odds > 2, "having the income missing more than doubles the default odds", "exp(coef / scale) for the indicator column; the scale is final[1].named_transformers_['num'][-1].scale_")
    check(top4_numeric is not None and set(top4_numeric) >= {"num__ratio", "num__missingindicator_income"}, "ratio and the income-missing indicator are among the four heaviest numerics", "sort by |coef| the names starting with 'num__'")
done()

## What you learned

- Ablation answers the question "what is this piece worth?" with a number. Here **the missingness indicators are worth about 0.06 of AUC, the categoricals about 0.04**:
  the most informative fact is that the income **was not declared**. If you just imputed, you would throw away the strongest signal in the dataset.
- A missing value is not always "noise to fill in": when it is missing for a reason linked to the target (`MNAR`), the absence is a feature. That is why `add_indicator=True`
  is almost free and almost always worth trying.
- Odds ratios are readable only if you know **in which units** they are: the coefficient of a standardized feature is "per one standard deviation", that of a 0/1 indicator must be reported in original
  units (dividing by the scale of the scaler) to say "income missing versus present". Comparing coefficients of standardized features and of dummies without this step (the occupation dummies
  end up in the top positions) is the classic way of telling false stories: it is the theme of D1.
- Look at the table of the numeric coefficients: `missingindicator_ratio` and `missingindicator_income` have the same coefficient (0.315) because they are **the same column** (the ratio is missing exactly when the
  income is missing). With a regularized logistic it does no harm, but with an unregularized linear model the perfect collinearity would be a problem: it is a case where `add_indicator` must be used with judgment.
- The whole construction sits in a single object that lives inside `cross_val_score`: imputation, scaler and one-hot are learned inside every fold, with no leakage.

## Common pitfall

Filling the missing values with the median **without indicators** and concluding that "income matters little": with the imputation you replaced the signal (whoever does not declare) with a
typical value equal for everyone. The AUC without indicators is noticeably lower, and no error tells you so.

## Interview question

> *Click on a question to show its answer.*

<details><summary>You have columns with missing values: how do you decide how to treat them?</summary>

First I understand **why** they are missing (random, linked to another column, linked to the target). If the mechanism is informative I add a missingness indicator
on top of a simple imputation (median or most frequent) inside the pipeline, and measure with an ablation in cross-validation how much it is worth. More
sophisticated imputations (KNN, iterative) are justified only if they really gain over the median plus the indicator.

</details>